# Rolling Risk Metrics

Use this notebook to develop rolling ETF fragility and risk diagnostics from the processed core panel.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config

core_panel = pd.read_csv(config.CORE_PANEL_CSV, parse_dates=["Date", "Inception"])
core_panel = core_panel.sort_values(["Symbol", "Date"])
core_panel.head()

In [ ]:
window = 26

rolling = core_panel[["Symbol", "Date", "RET_XS"]].copy()
rolling["rolling_vol_26w"] = (
    rolling.groupby("Symbol")["RET_XS"]
    .rolling(window=window, min_periods=12)
    .std()
    .reset_index(level=0, drop=True)
)
rolling["rolling_var_10_26w"] = (
    rolling.groupby("Symbol")["RET_XS"]
    .rolling(window=window, min_periods=12)
    .quantile(0.10)
    .reset_index(level=0, drop=True)
)

rolling.tail()

In [ ]:
latest = rolling.dropna().sort_values("Date").groupby("Symbol").tail(1)
latest.sort_values("rolling_vol_26w", ascending=False).head(20)